## How Spark Accumulators Work

### The Core Mechanism

When you create an accumulator with `sc.accumulator(0)`, Spark uses a **driver-worker protocol** built on top of its task execution lifecycle:

```python
Driver                          Workers
  |                               |
  |-- broadcast accumulator ref ->|
  |                               | (tasks run, update local copies)
  |<-- task completion messages --|
  |   (each carries delta values) |
  |                               |
  | [driver merges all deltas]    |
  |                               |
  | acc.value  ← safe to read now |
```

### Step-by-Step

**1. Registration on the Driver**
When you call `sc.accumulator(0)`, the accumulator is registered in the driver's `AccumulatorContext` with a unique ID. Workers never hold the authoritative value — the driver does.

**2. Serialized into Tasks**
When a job is submitted, each task gets a serialized reference to the accumulator (by ID + initial value). Workers operate on **local copies**, not the shared object.

**3. Workers Send Deltas Back**
When a task finishes, the worker doesn't just return the RDD result — it sends back a `TaskResult` object that includes:
- The computed partition data
- **A list of `(accumulator_id → delta)` pairs** for every accumulator updated

**4. Driver Merges After All Tasks Complete**
The `DAGScheduler` on the driver collects results from **all tasks in a stage** before marking that stage complete. As each `TaskResult` arrives, it calls `Accumulable.merge(delta)` on the driver side.

Only after **all tasks report back** does Spark proceed, so by the time your next line of code reads `acc.value`, every worker's contribution is guaranteed to be merged.

---

### Why You Should Only Read Accumulators After an Action

```python
rdd = sc.parallelize([1, 2, 3, 4, 5])
counter = sc.accumulator(0)

rdd.foreach(lambda x: counter.add(x))

# ✅ Safe: foreach is an action, job is fully complete
print(counter.value)  # 15

# ❌ Unsafe: map is a transformation, lazy — job hasn't run yet
rdd.map(lambda x: counter.add(x))
print(counter.value)  # Still 0!
```

---

### The "Double Counting" Gotcha

Spark **does not guarantee exactly-once accumulator updates** in all cases. If a task is re-executed due to failure or speculation, the accumulator gets updated **twice**:

```python
# If task retries happen, you might see > expected value
counter = sc.accumulator(0)
rdd.foreach(lambda x: counter.add(1))
# counter.value might be > rdd.count() if retries occurred
```

Spark only guarantees **exactly-once for actions on the same lineage**, but not for failed/speculative tasks. This is why accumulators are best used for **diagnostics/debugging** (counters, logging), not for correctness-critical computations.

---

### Summary

| Concern | How Spark handles it |
|---|---|
| "Does the driver know about all workers?" | Yes — the DAGScheduler tracks every task in a stage |
| "Are updates guaranteed before `acc.value`?" | Yes — only after all tasks in the action's job complete |
| "Is it safe from race conditions?" | Yes — merging happens sequentially on the driver thread |
| "Is it exactly-once?" | No — retried/speculative tasks can double-count |